In [47]:
import pygame
import random
import math
import os

# --- KHỞI TẠO ---
pygame.init()
pygame.mixer.init() 
clock = pygame.time.Clock()
WIDTH, HEIGHT = 800, 450
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption('DINO GAME')

# Cấu hình logic game
MAX_SPEED = 15.0
SPEED_ADD = 0.5 
WIN_SCORE = 50.0

# Màu sắc
PINK = (255, 105, 180)
BLACK = (50, 50, 50)
WHITE = (255, 255, 255)
RED = (255, 0, 0)
GOLD = (255, 215, 0)
CYAN = (0, 255, 255)

# Cấu hình làn đường
AI_GROUND = 180
PLAYER_GROUND = 380

# --- LOAD ASSETS ---
def get_img(path, color, size=(40, 40)):
    try:
        full_path = os.path.join('dino', 'assets', path)
        return pygame.image.load(full_path).convert_alpha()
    except:
        surf = pygame.Surface(size); surf.fill(color)
        return surf

try:
    pygame.mixer.music.load(os.path.join('dino', 'sound', 'nhac_nen.mp3'))
    pygame.mixer.music.set_volume(0.8)
    jump_tree_sound = pygame.mixer.Sound(os.path.join('dino', 'sound', 'te.wav'))   
    eat_fruit_sound = pygame.mixer.Sound(os.path.join('dino', 'sound', 'tick.wav')) 
    win_sound = pygame.mixer.Sound(os.path.join('dino', 'sound', 'victory_game.mp3')) 
    lose_sound = pygame.mixer.Sound(os.path.join('dino', 'sound', 'lost_game.mp3'))
except:
    jump_tree_sound = eat_fruit_sound = win_sound = lose_sound = None

bg_img = get_img('background.jpg', WHITE, (WIDTH, HEIGHT))
bg_img = pygame.transform.scale(bg_img, (WIDTH, HEIGHT))
tree_img = get_img('tree.png', (34, 139, 34), (30, 50))
dino_player_img = get_img('dinosaur.png', PINK)
dino_ai_img = get_img('dinoAI.png', BLACK)
fruit_img = get_img('fruit.png', (128, 0, 128), (20, 20))

# --- CẤU HÌNH FONT ---
try:
    font_path = os.path.join('dino', '04B_19.TTF')
    game_font = pygame.font.Font(font_path, 22)
    # Font lớn hơn cho thông báo kết quả (Cỡ 60)
    result_font = pygame.font.Font(font_path, 60)
except:
    game_font = pygame.font.SysFont('Arial', 24, bold=True)
    result_font = pygame.font.SysFont('Arial', 70, bold=True)

def reset_game():
    if win_sound: win_sound.stop()
    if lose_sound: lose_sound.stop()
    pygame.mixer.music.play(-1) 
    return {
        'p_y': PLAYER_GROUND, 'p_vel': 0, 'p_jump': False,
        'ai_y': AI_GROUND, 'ai_vel': 0, 'ai_jump': False,
        'p_obj': {'x': WIDTH, 'type': random.choice(['cactus', 'fruit']), 'passed': False},
        'ai_obj': {'x': WIDTH, 'type': random.choice(['cactus', 'fruit']), 'passed': False},
        'p_score': 0.0, 'ai_score': 0.0,
        'p_speed': 5.0, 'ai_speed': 5.0,
        'active': False, 'status': None, 'particles': []
    }

data = reset_game()
pygame.mixer.music.stop()

def create_fireworks(x, y):
    colors = [PINK, GOLD, CYAN, RED, WHITE]
    chosen_color = random.choice(colors)
    for _ in range(30):
        angle = random.uniform(0, 2 * math.pi)
        speed = random.uniform(2, 7)
        data['particles'].append([x, y, math.cos(angle)*speed, math.sin(angle)*speed, chosen_color, random.randint(30, 60)])

def update_particles():
    for p in data['particles'][:]:
        p[0] += p[2]; p[1] += p[3]; p[3] += 0.1; p[5] -= 1
        if p[5] <= 0:
            data['particles'].remove(p)
            continue
        size = max(1, int(p[5] / 15))
        pygame.draw.circle(screen, p[4], (int(p[0]), int(p[1])), size)

def draw_ui():
    # Điểm và Tốc độ thông thường
    screen.blit(game_font.render(f'YOU: {data["p_score"]:.1f} / {WIN_SCORE}', True, PINK), (30, 20))
    screen.blit(game_font.render(f'SPEED: {data["p_speed"]:.1f}', True, PINK), (30, 50))
    screen.blit(game_font.render(f'AI: {data["ai_score"]:.1f} / {WIN_SCORE}', True, BLACK), (WIDTH - 250, 20))
    screen.blit(game_font.render(f'SPEED: {data["ai_speed"]:.1f}', True, BLACK), (WIDTH - 250, 50))

    if not data['active']:
        overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
        overlay.fill((255, 255, 255, 180)) 
        screen.blit(overlay, (0,0))
        
        if data['status'] is None: 
            m2 = game_font.render("PRESS SPACE TO START", True, BLACK)
            screen.blit(m2, (WIDTH//2 - m2.get_width()//2, HEIGHT//2))
        else: 
            # THAY ĐỔI TẠI ĐÂY: Dùng result_font cỡ lớn và màu RED cho cả thắng/thua
            msg = result_font.render(data['status'], True, RED)
            retry = game_font.render("PRESS SPACE TO RESTART!", True, BLACK)
            
            # Căn chỉnh chữ ra giữa màn hình
            screen.blit(msg, (WIDTH//2 - msg.get_width()//2, HEIGHT//2 - 70))
            screen.blit(retry, (WIDTH//2 - retry.get_width()//2, HEIGHT//2 + 20))

# --- VÒNG LẶP CHÍNH ---
run = True
while run:
    screen.blit(bg_img, (0, 0)) 
    pygame.draw.line(screen, (200, 200, 200), (0, AI_GROUND), (WIDTH, AI_GROUND), 2)
    pygame.draw.line(screen, (200, 200, 200), (0, PLAYER_GROUND), (WIDTH, PLAYER_GROUND), 2)

    for event in pygame.event.get():
        if event.type == pygame.QUIT: run = False
        if event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE:
            if not data['active']: 
                data = reset_game(); data['active'] = True 
            elif not data['p_jump']: 
                data['p_vel'] = -13; data['p_jump'] = True

    if data['active']:
        data['p_obj']['x'] -= data['p_speed']
        data['ai_obj']['x'] -= data['ai_speed']

        if data['p_obj']['x'] < -50:
            data['p_obj'] = {'x': WIDTH + random.randint(100, 400), 'type': random.choice(['cactus', 'fruit']), 'passed': False}
        if data['ai_obj']['x'] < -50:
            data['ai_obj'] = {'x': WIDTH + random.randint(100, 400), 'type': random.choice(['cactus', 'fruit']), 'passed': False}

        for side, ground in [('p', PLAYER_GROUND), ('ai', AI_GROUND)]:
            if data[f'{side}_jump']:
                data[f'{side}_y'] += data[f'{side}_vel']
                data[f'{side}_vel'] += 0.85
                if data[f'{side}_y'] >= ground:
                    data[f'{side}_y'] = ground; data[f'{side}_jump'] = False

        if data['ai_obj']['type'] == 'cactus' and 0 < data['ai_obj']['x'] - 120 < 115 + (data['ai_speed'] * 1.2) and not data['ai_jump']:
            data['ai_vel'] = -12.5; data['ai_jump'] = True

        rect_p_obj = screen.blit(tree_img if data['p_obj']['type'] == 'cactus' else fruit_img, (data['p_obj']['x'], PLAYER_GROUND - 40))
        fruit_y_ai = AI_GROUND - 30 if data['ai_obj']['type'] == 'fruit' else AI_GROUND - 40
        rect_ai_obj = screen.blit(tree_img if data['ai_obj']['type'] == 'cactus' else fruit_img, (data['ai_obj']['x'], fruit_y_ai))
        p_rect = screen.blit(dino_player_img, (150, data['p_y'] - 40))
        ai_rect = screen.blit(dino_ai_img, (120, data['ai_y'] - 40))

        if data['p_obj']['type'] == 'cactus':
            if p_rect.colliderect(rect_p_obj): 
                data['active'] = False; data['status'] = "YOU LOSE!"
                pygame.mixer.music.stop()
                if lose_sound: lose_sound.play()
            elif data['p_obj']['x'] < 150 and not data['p_obj']['passed']:
                data['p_score'] += 1.0; data['p_obj']['passed'] = True
                if jump_tree_sound: jump_tree_sound.play()
        elif data['p_obj']['type'] == 'fruit' and p_rect.colliderect(rect_p_obj):
            data['p_score'] += 1.0; data['p_obj']['x'] = -100
            data['p_speed'] = min(MAX_SPEED, data['p_speed'] + SPEED_ADD)
            if eat_fruit_sound: eat_fruit_sound.play()

        if data['ai_obj']['type'] == 'cactus' and data['ai_obj']['x'] < 120 and not data['ai_obj']['passed']:
            data['ai_score'] += 1.0; data['ai_obj']['passed'] = True
        if data['ai_obj']['type'] == 'fruit' and ai_rect.colliderect(rect_ai_obj):
            data['ai_score'] += 1.0; data['ai_obj']['x'] = -100
            data['ai_speed'] = min(MAX_SPEED, data['ai_speed'] + SPEED_ADD)

        if data['p_score'] >= WIN_SCORE: 
            data['active'] = False; data['status'] = "YOU WIN!"
            pygame.mixer.music.stop()
            if win_sound: win_sound.play()
        elif data['ai_score'] >= WIN_SCORE: 
            data['active'] = False; data['status'] = "YOU LOSE!"
            pygame.mixer.music.stop()
            if lose_sound: lose_sound.play()

    if data['status'] == "YOU WIN!":
        if random.random() < 0.15: 
            create_fireworks(random.randint(100, 700), random.randint(50, 200))

    update_particles()
    draw_ui()
    pygame.display.update()
    clock.tick(60)

pygame.quit()